# 🎯 [K15_M04] 검색 결과를 예약표와 대조하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m04_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 04. 검색 결과를 예약표와 대조하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 04. 검색 결과를 예약표와 대조하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>AI가 제시한 빈 시설 검색 결과가 실제 가상 부대 예약표와 일치하는지 육안으로 대조하여 사실관계를 철저히 검증합니다. 비어 있다고 답변한 칸을 하나씩 원본 표와 비교하여 정상, 오류, 누락 여부를 판별하고, 점검·소독 구간을 빈 곳으로 오인하지 않았는지 집중 확인합니다. 답변마다 근거 행(시설·요일·시간)을 명시하도록 유도하여 인간 검증(HITL) 체계를 확립합니다.</p>

**[학습 목표 및 체득 역량]**
<p>AI가 추출한 빈 시설 결과를 원본 예약표와 대조하여 일치·오류·누락 여부를 판별하고, 점검·소독 오인 여부와 근거 행 표기 여부를 점검합니다. <strong>이를 통해 AI 산출물을 무비판적으로 수용하지 않고 원문 대조를 통해 사실관계를 직접 확인하는 인간 검증(HITL) 및 팩트체크 기법을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: AI가 말한 빈 칸을 원본 표에서 찾는다.
<h4>이 단계의 역할</h4><p>미션3에서 AI가 '비어 있다'고 답변한 모든 시설/시간대 슬롯을 원본 28행 예약 현황표에서 직접 찾아 1:1 위치를 추적합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>AI 답변에서 언급된 '시설명', '요일', '시간대'를 추출하여 대조표 좌측 열에 나열합니다.</p></li>
<li><p>원본 예약표에서 해당 행과 열을 찾아 실제 텍스트가 '빈'인지, 다른 교육 일정이 적혀 있는지 확인합니다.</p></li>
<li><p>특히 강당 금요일 11-13(음향 점검)이나 체력단련실 수요일 14-16(소독) 칸이 빈 칸으로 잘못 분류되지 않았는지 위치를 정밀 검증합니다.</p></li>
</ol>
<h4>💡 AI 근거 위치 추출 질의 예시</h4><blockquote><p><strong>[근거 슬롯 위치 추출 질의]:</strong> "네가 앞서 답변한 빈 시설 목록 각각에 대해, 참조한 [예약 현황표]의 정확한 행(시설명, 요일)과 열(시간대 슬롯)의 원본 텍스트를 그대로 복사해 표로 보여줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>좌표 일치:</strong> AI가 지목한 칸이 원본 표의 행/열과 정확히 일치하는가?</li>
<li><strong>원문 텍스트 대조:</strong> 원본 셀에 적힌 글자(예: '빈', '3중대 교육', '점검')를 왜곡 없이 가져왔는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>AI 답변 슬롯과 원본 예약표 간 1:1 대조 위치 매핑 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>1:1 원본 위치 매핑 작업표</p></li></ul>

#### 🔹 Step 2: 맞음·틀림·누락을 표시한다.
<h4>이 단계의 역할</h4><p>AI의 답변 결과를 원본 표와 비교하여 '맞음(일치)', '틀림(오류)', '누락(빠뜨림)'으로 판정하는 대조 검증표를 완성합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>대조 검증표 서식에 [질문 번호, AI 답변, 원본 상태, 판정(맞음/틀림/누락), 상세 사유]를 기록합니다.</p></li>
<li><p>실제 비어 있는데 AI가 언급하지 않은 슬롯이 있다면 '누락'으로 판정합니다.</p></li>
<li><p>점검이나 소독 칸을 빈 칸으로 잘못 안내한 사례가 있는지 집중 검토하여 오류를 잡아냅니다.</p></li>
</ol>
<h4>💡 AI 답변 오류 자가 분석 질의 예시</h4><blockquote><p><strong>[교차 검증 질의]:</strong> "내가 원본 표를 확인해 보니 수요일 14-16시 체력단련실은 '소독(이용 불가)'인데, 너의 답변에서 이를 빈 칸으로 안내했는지 여부를 스스로 확인하고 오류 원인을 분석해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>판정 객관성:</strong> 맞음/틀림/누락 판정이 주관적이지 않고 원본 표의 팩트에 근거했는가?</li>
<li><strong>점검·소독 오인 적출:</strong> 이용 불가 칸을 빈 칸으로 오인한 치명적 행정 실수를 100% 잡아냈는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>검색 결과 대조 검증표 작성 완료 및 판정 내역 증빙</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>검색 결과 대조 검증표 (맞음·틀림·누락 표기)</p></li></ul>

#### 🔹 Step 3: 답변에 근거 행을 쓰게 한다.
<h4>이 단계의 역할</h4><p>AI가 답변을 출력할 때 반드시 원본 표의 '근거 행(시설·요일·시간대)'을 의무적으로 명시하도록 프롬프트를 개선합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>기존 프롬프트에 '답변 끝에 반드시 근거가 된 [시설명 / 요일 / 시간대 / 원본 셀 내용]을 표기하라'는 지침을 추가합니다.</p></li>
<li><p>동일한 질문 4개를 다시 실행하여 답변마다 근거 행이 100% 명시되는지 확인합니다.</p></li>
<li><p>근거 행이 표기됨으로써 사용자가 원본 표를 신속하게 재검증할 수 있는 업무 효율성을 평가합니다.</p></li>
</ol>
<h4>💡 AI 근거 명시 프롬프트 예시</h4><blockquote><p><strong>[근거 행 의무화 질의]:</strong> "[기존 프롬프트 지침에 추가]<br>※ 출력 규칙: 답변 마지막 줄에 반드시 근거 행을 다음 형식으로 작성하라.<br>[근거]: 시설명 | 요일 | 시간대 | 원본 표 상태(예: 빈, 점검)"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>근거 표기율:</strong> 4개 질문 답변 모두에서 예외 없이 근거 행이 명시되었는가?</li>
<li><strong>근거 신뢰도:</strong> 명시된 근거 행 정보가 실제 원본 28행 예약표와 오차 없이 일치하는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>근거 행 표기 준수율 100% 달성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>근거 행 표기 규칙이 반영된 개선 프롬프트 및 실행 결과</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>'빈 칸만 안내해 줘'와 '이용 불가 칸(점검·소독)도 함께 알려 줘' 지시문 간의 답변 품질 차이를 분석하는 심화 과제를 수행합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>프롬프트 지시를 변경하여 가용 빈 칸뿐만 아니라 동 시간대 이용 불가 칸의 사유까지 함께 출력하도록 요청합니다.</p></li>
<li><p>이용 불가 사유(예: 음향 점검, 소독)가 명시될 때 사용자가 얻는 행정적 안전성을 평가합니다.</p></li>
<li><p>질의 결과 전후를 비교하여 정보 완전성을 정리합니다.</p></li>
</ol>
<h4>💡 AI 비교 심화 질의 예시</h4><blockquote><p><strong>[이용 불가 병기 질의]:</strong> "[개선 프롬프트] 금요일 대대 강당의 이용 가능 여부를 알려주되, 빈 시간대뿐만 아니라 '이용 불가(점검/행사)' 시간대와 그 구체적 사유도 함께 구분하여 안내해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>사유 정확도:</strong> 금요일 11-13이 '음향 점검'으로 정확히 구분되었는가?</li>
<li><strong>정보 명확성:</strong> 장병 입장에서 왜 이용할 수 없는지 납득 가능한 정보가 제공되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>이용 불가 사유 병기 비교 결과 보고서</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 비교 질의 결과 및 분석 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 검색 결과 원본 대조 검증표 (HITL 체크리스트)
| 질문 번호 | AI가 안내한 빈 슬롯 | 원본 표 실제 상태 | 판정 (맞음 / 틀림 / 누락) | 점검·소독 오인 여부 |
| :--- | :--- | :--- | :--- | :--- |
| 질문 1 | 대대 강당 수요일 19-21 | '빈' | 맞음 | 정상 (14-16과 혼동 없음) |
| 질문 2 | 세미나실 A 목요일 14-16, 16-18<br>세미나실 B 목요일 14-16 | 각 셀 '빈' | 맞음 | 정상 |
| 질문 3 | 금요일 오전 이용 불가 안내 | 09-11 조회, 11-13 점검 | 맞음 | 정상 (점검 칸 이용불가 판별) |
| 질문 4 | 종일 비는 시설 없음 | 각 시설 1개 이상 일정 있음 | 맞음 | 정상 |

## 2. 근거 행(Origin Row) 의무 표기 프롬프트 지침
```markdown
※ 답변 규칙:
답변의 마지막에 반드시 참조한 표의 [시설명 | 요일 | 시간대 | 상태]를 명시할 것.
예시:
[답변]: 수요일 저녁 대대 강당은 이용 가능합니다.
[근거]: 대대 강당 | 수요일 | 19-21 | 상태: 빈
```

## 3. 심화 비교: 이용 불가 사유 병기 유무 비교
- **빈 칸만 알려준 경우**: "금요일 오전 강당은 비어 있지 않습니다." (다소 불친절하고 원인 불명)
- **이용 불가 사유 병기 시**: "09-11시는 월례 조회, 11-13시는 음향 정기점검으로 이용 불가합니다." (행정 신뢰도 극대화)

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] AI 결과 검증·HITL 체계 — 검색 결과를 원본 표와 1:1 대조하여 오류/누락을 적발하고 근거 행 표기 규칙을 확립했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>원본 예약표 1:1 대조 검증표(맞음·틀림·누락 판정) 작성 완료</p></li><li><p>점검 및 소독 슬롯을 빈 칸으로 오인하는 행정 오류 100% 적발</p></li><li><p>근거 행(Origin Row) 표기 준수율 100% 달성</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>AI 결과를 무비판적으로 수용하지 않고 인간이 직접 팩트체크하여 오류를 찾아내는 인간 검증(HITL) 역량을 갖추었는지 대조표를 확인한다.</p></blockquote>


In [ ]:
# [미션 04] 프롬프트 및 실습 결과 기록
mission_04_prompt = """# [미션 04] 근거 행 의무 표기 및 HITL 원본 대조 프롬프트
너는 가상 한빛대대 시설 정보 검색 도우미다.
반드시 아래 [예약 현황표]에 명시된 내용만을 근거로 답하라.
모든 답변의 끝에는 반드시 참조한 원본 데이터의 [시설명 | 요일 | 시간대 | 상태]를 명시하라.
또한 이용 불가인 경우 단순 거절에 그치지 않고 구체적 사유(예약 부서명 또는 점검 내용)를 병기하라.

[예약 현황표]
(28행 마크다운 표 입력)

[질문]
금요일 오전에 강당 쓸 수 있나요?
"""

mission_04_result = """### 📋 검색 결과 원본 대조 검증표 (HITL) 및 근거 표기

#### 1. 검색 결과 원본 대조 검증표
| 질문 번호 | AI 안내 가용 상태 | 원본 표 실제 상태 | 판정 | 점검·소독 오인 여부 |
| :---: | :--- | :--- | :---: | :--- |
| 질문 1 | 대대 강당 수요일 19-21 빈 | '빈' | 맞음 | 정상 (14-16과 혼동 없음) |
| 질문 2 | 세미나실 A 목요일 14-18, B 14-16 빈 | 각 셀 '빈' | 맞음 | 정상 (B 16-21 AI스터디 제외) |
| 질문 3 | 금요일 오전 강당 이용 불가 | 09-11 조회, 11-13 점검 | 맞음 | 정상 (점검 칸 이용불가 판별) |
| 질문 4 | 종일 비는 시설 없음 | 각 시설 1개 이상 일정 있음 | 맞음 | 정상 |

#### 2. 근거 행(Origin Row) 표기 및 이용 불가 사유 병기 효과
- **AI 출력 예시**:
  - [답변]: 금요일 오전 대대 강당은 이용 불가합니다.
  - [사유]: 09-11시는 '월례 조회', 11-13시는 '점검(음향)' 일정이 배정되어 있습니다.
  - [근거 행]: 대대 강당 | 금 | 09-11 | 월례 조회 / 대대 강당 | 금 | 11-13 | 점검(음향)
- **효과 분석**: 단순 '불가' 안내 대비 구체적 사유와 원본 행을 명시함으로써 이용 부서와의 불필요한 실랑이를 차단하고 행정 신뢰도를 대폭 향상시킴.
"""

print(f"미션 04 실습 기록 완료: 프롬프트 {len(mission_04_prompt.strip())}자, 결과 {len(mission_04_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션4] AI 답변 슬롯 1:1 원본 대조 검증 함수
def verify_slot(facility, day, time_slot):
    match = df_schedule[(df_schedule['facility'] == facility) & (df_schedule['day'] == day)]
    if match.empty:
        return "시설/요일 정보 없음"
    val = match[time_slot].values[0]
    is_available = (val == "빈")
    return {"원본 상태": val, "이용 가능 여부": "가능" if is_available else "불가"}

# 테스트 검증: 금요일 강당 11-13 (점검 칸)
print("금요일 강당 11-13 대조:", verify_slot("대대 강당", "금", "11-13"))
# 테스트 검증: 수요일 체력단련실 14-16 (소독 칸)
print("수요일 체력단련실 14-16 대조:", verify_slot("체력단련실(단체)", "수", "14-16"))
